## Image Enhancement Analysis & Pipeline Selection

During the enhancement stage on the **Field-PDD** leaf lesion dataset, our goal was to improve visual clarity, illuminate shadowed regions on leaf surfaces, and highlight lesion boundaries and leaf veins without distorting natural plant colors or introducing distracting artifacts.

### 1. Why Gamma Correction with `gamma < 1` (`gamma = 0.7`)?
In power-law (Gamma) transformation, the brightness curve responds differently depending on whether `gamma` is less than or greater than `1`:
* **When `gamma < 1` (specifically `gamma = 0.7` in our pipeline):** The mapping curve bends upward in the darker intensity ranges. This **brightens shadow and low-light regions** on the leaf surface while keeping brighter areas controlled and natural, making dark or shaded leaf sections much easier to inspect without blowing out highlights.
* **When `gamma > 1`:** The curve bends downward and darkens mid-tones and shadows, which would make under-lit or shadowed leaf regions even harder to read.

Therefore, setting **`gamma = 0.7` (`gamma < 1`)** was the right choice to lift shadows and establish a clearer baseline illumination before local contrast enhancement.

### 2. Comparing CLAHE vs. Unsharp Masking (and Other Alternatives)
To apply **at least two enhancement steps** and select the most effective secondary technique, we tested additional methods on validation (`val`) images and compared them against the original images (`OriginalData`) as well as alongside Gamma correction:
* **Unsharp Masking:** We tested Unsharp Masking across settings (`amount = 0.5` and `amount = 1.2`). At low settings (`0.5`), the visual difference on leaf lesions was barely noticeable; at higher settings (`1.2`), it sharpened edges/noise but could accentuate fine grain rather than improving overall local contrast inside the lesion and leaf tissue.
* **Bilateral Filter:** Helpful for smoothing flat regions while preserving edges, but it does not boost local contrast inside subtle lesion patterns on its own.
* **CLAHE on the L Channel (`LAB` Color Space, `clipLimit = 2.0`, `tileGridSize = (8, 8)`):** Compared directly against **Unsharp Masking**, **CLAHE on the `L` (luminance) channel** proved more effective for these leaf images. By applying contrast-limited adaptive histogram equalization only to the lightness channel (`L`) while keeping the color channels (`a` and `b`) in `LAB` space intact, it enhances **local contrast** across small regions—bringing out subtle variations in diseased spots and leaf veins—without shifting natural leaf colors or over-amplifying noise.

### 3. Why We Combined `Gamma (0.7)` + `CLAHE (L, clipLimit = 2.0)` Together
Using **Gamma Correction (`0.7`) followed by CLAHE (`clipLimit = 2.0` on the `L` channel)** works as a complementary two-stage pipeline:
1. **Stage 1 — Shadow Illumination Lift (`Gamma = 0.7`):** Opens up dark and shadowed areas across the leaf so mid-tones and lesion regions are clearer.
2. **Stage 2 — Local Contrast Refinement (`CLAHE on L`):** Enhances fine local contrast in each tile (`8x8`) up to a safe contrast limit (`clipLimit = 2.0`), making lesion textures and leaf details clearer while preserving color fidelity (`LAB` color space).

### 4. Additional Tested Filters vs. Original (`Laplacian`, `High-pass`, `Gaussian Blur`, `Median Blur`)
We also evaluated **Laplacian sharpening**, **High-pass sharpening**, **Gaussian Blur**, and **Median Blur** individually against `OriginalData`:
* **Minimal impact on clean leaf images:** When tested directly against the original leaf images, these filters made little to no meaningful improvement for our lesion visibility goal.
* **Why `Median Blur` had limited effect without `salt-and-pepper` noise:** Median filtering is primarily effective when an image contains **salt-and-pepper noise** (isolated bright and dark pixel spikes). Since the leaf images in our sample do not suffer noticeably from salt-and-pepper noise, applying `Median Blur` directly to the clean images did not produce a clear benefit over the original images.
* **Conclusion for the final pipeline:** Because these methods added minimal clarity advantage compared to **Gamma (`0.7`) + CLAHE on the `L` channel (`clipLimit = 2.0`)**, we kept **Gamma + CLAHE** as our final enhancement pipeline.


In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

extract_dir = Path("/content/field_pdd_extracted")
orig_root = extract_dir / "OriginalData"
exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

orig_by_rel = {}
orig_by_name = {}
for p in orig_root.rglob("*"):
    if p.is_file() and p.suffix.lower() in exts:
        rel = str(p.relative_to(orig_root)).replace("\\", "/")
        orig_by_rel[rel] = p
        orig_by_name.setdefault(p.name, p)

split_candidates = list(Path("/content").rglob("split.csv")) + list(Path("/content/drive/MyDrive").rglob("split.csv"))
split_df = pd.read_csv(split_candidates[0])
val_df = split_df[split_df["split"].astype(str).str.lower() == "val"]

val_paths = []
for rel in val_df["image"].astype(str):
    rel_clean = rel.strip().lstrip("./")
    fname = Path(rel_clean).name
    if rel_clean in orig_by_rel:
        val_paths.append(orig_by_rel[rel_clean])
    elif fname in orig_by_name:
        val_paths.append(orig_by_name[fname])

def only_laplacian(rgb: np.ndarray, weight: float = 0.4) -> np.ndarray:
    lap = cv2.Laplacian(rgb, cv2.CV_32F, ksize=3)
    out = rgb.astype(np.float32) - weight * lap
    return np.clip(out, 0, 255).astype(np.uint8)

def only_highpass(rgb: np.ndarray, sigma: float = 3.0, amount: float = 0.8) -> np.ndarray:
    low_pass = cv2.GaussianBlur(rgb, (0, 0), sigmaX=sigma, sigmaY=sigma).astype(np.float32)
    high_pass = rgb.astype(np.float32) - low_pass
    out = rgb.astype(np.float32) + amount * high_pass
    return np.clip(out, 0, 255).astype(np.uint8)

def only_gaussian(rgb: np.ndarray, ksize: tuple = (5, 5), sigma: float = 1.2) -> np.ndarray:
    return cv2.GaussianBlur(rgb, ksize, sigmaX=sigma, sigmaY=sigma)

def only_median(rgb: np.ndarray, ksize: int = 5) -> np.ndarray:
    return cv2.medianBlur(rgb, ksize)

def luma_stats(rgb: np.ndarray) -> tuple[float, float]:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return float(gray.mean()), float(gray.std())

methods = [
    ("Laplacian (w=0.4)", only_laplacian),
    ("High-pass (sigma=3.0, amt=0.8)", only_highpass),
    ("Gaussian Blur (5x5, sigma=1.2)", only_gaussian),
    ("Median Blur (ksize=5)", only_median),
]

rows = []
for name, fn in methods:
    m_orig, s_orig, m_proc, s_proc = [], [], [], []
    for p in val_paths:
        bgr = cv2.imread(str(p))
        if bgr is None:
            continue
        orig = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        proc = fn(orig)
        mo, so = luma_stats(orig)
        mp, sp = luma_stats(proc)
        m_orig.append(mo); s_orig.append(so)
        m_proc.append(mp); s_proc.append(sp)
    rows.append({
        "Method": name,
        "Orig Mean": round(float(np.mean(m_orig)), 2),
        "Proc Mean": round(float(np.mean(m_proc)), 2),
        "Orig Std": round(float(np.mean(s_orig)), 2),
        "Proc Std": round(float(np.mean(s_proc)), 2),
    })
display(pd.DataFrame(rows))

for p in val_paths[:1]:
    orig = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    for title, fn in methods:
        proc = fn(orig)
        fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
        axes[0].imshow(orig)
        axes[0].set_title(f"Original ({p.name})")
        axes[0].axis("off")
        axes[1].imshow(proc)
        axes[1].set_title(title)
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()


### 5. Synthetic Salt-and-Pepper Noise Experiment & Median Restoration
To verify how `Median Blur` behaves when salt-and-pepper noise is present, we added synthetic salt-and-pepper noise to an original image, restored it with `Median Blur`, and optionally applied our `Gamma (0.7) + CLAHE` enhancement step afterward. *(This experiment is for demonstration only and is not applied to the main dataset.)*


In [ ]:
def add_salt_and_pepper(rgb: np.ndarray, prob: float = 0.10, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    noisy = rgb.copy()
    rnd = rng.random(rgb.shape[:2])
    noisy[rnd < prob / 2] = 0
    noisy[rnd > 1.0 - prob / 2] = 255
    return noisy

def gamma_and_clahe(rgb: np.ndarray, gamma: float = 0.7, clip_limit: float = 2.0, grid_size: tuple = (8, 8)) -> np.ndarray:
    inv_gamma = 1.0 / gamma
    lut = np.array([np.clip(((i / 255.0) ** inv_gamma) * 255.0, 0, 255) for i in range(256)], dtype=np.uint8)
    g_img = cv2.LUT(rgb, lut)
    lab = cv2.cvtColor(g_img, cv2.COLOR_RGB2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    l_eq = clahe.apply(l_ch)
    return cv2.cvtColor(cv2.merge((l_eq, a_ch, b_ch)), cv2.COLOR_LAB2RGB)

orig = cv2.cvtColor(cv2.imread(str(val_paths[0])), cv2.COLOR_BGR2RGB)
prob = 0.10
ksize = 7
noisy = add_salt_and_pepper(orig, prob=prob, seed=42)
denoised = cv2.medianBlur(noisy, ksize)
denoised_enhanced = gamma_and_clahe(denoised, gamma=0.7, clip_limit=2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
panels = [
    (orig, "1. Original"),
    (noisy, f"2. + Salt & Pepper ({int(prob*100)}%)"),
    (denoised, f"3. Treated (Median k={ksize})"),
    (denoised_enhanced, "4. Median + Gamma + CLAHE"),
]
for ax, (im, title) in zip(axes, panels):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()
